### **What is SQL**?

SQL (Structured Query Language) is a standard language for managing and manipulating relational databases. It's used to perform operations like creating, reading, updating, and deleting data.

### **Relational Databases**

A relational database is a type of database that organizes data into tables with rows and columns. These tables can be related to each other through common fields, allowing for efficient data retrieval and management. Examples include MySQL, PostgreSQL, SQLite, and Oracle.

### **Non-relational Databases**

Non-relational databases (also known as NoSQL databases) are databases that do not use the tabular schema of rows and columns found in traditional relational databases. They offer more flexibility and scalability for certain types of data and applications. Examples include MongoDB, Cassandra, and Redis.

### **Software Used for SQL**

Various software tools are used to interact with SQL databases, including:

*   **Database Management Systems (DBMS):** Software like MySQL, PostgreSQL, SQL Server, Oracle, and SQLite that manage the database itself.
*   **SQL Clients:** Tools like DBeaver, pgAdmin, MySQL Workbench, and command-line interfaces that allow users to connect to and interact with databases.
*   **Programming Languages:** Many programming languages (like Python, Java, PHP) have libraries and connectors to interact with SQL databases.


### **What is SQL Injection**?

**SQL Injection** is a web security vulnerability that allows an attacker to interfere with the queries that an application makes to its database. It generally allows an attacker to view data that they are not normally able to retrieve. This might include data belonging to other users, or any other data that the application itself is able to access. In many cases, an attacker can modify or delete this data, causing persistent changes to the application's content or behavior.

In essence, an attacker provides malicious SQL code as input to an application, which is then executed by the database. This can happen when an application uses user input directly in SQL queries without proper sanitization or parameterization, as demonstrated in the `vulnerable_login` function in this notebook.

In [1]:
#Import Libs
import sqlite3

In [2]:
#Create a Database in Memory - Example
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

In [3]:
#Create a Table
cursor.execute('''
CREATE TABLE users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT NOT NULL,
    password TEXT NOT NULL
)
''')

In [4]:
#Create Users and Passwords
users = [
    ('admin', 'admin123'),
    ('alice', 'alicepass'),
    ('bob', 'bobpass')
]

In [5]:
#Add the Users and Passwords in the Table
cursor.executemany('INSERT INTO users (username, password) VALUES (?, ?)', users)

In [6]:
#Show the Table Users
cursor.execute("SELECT * FROM users")
rows = cursor.fetchall()

for row in rows:
    print(row)

(1, 'admin', 'admin123')
(2, 'alice', 'alicepass')
(3, 'bob', 'bobpass')


In [7]:
#Commit the Table
conn.commit()

In [8]:
#Create a Function - Vulnerable Login
def vulnerable_login(username, password):
    query = f"SELECT * FROM users WHERE username = '{username}' AND password = '{password}'"
    print(f"Execute SQL:\n{query}\n")

    cursor.execute(query)
    result = cursor.fetchone()

    if result:
        print("✅ Success! Login with User:", result)
    else:
        print("❌ Failed!")

In [9]:
# Login Test
vulnerable_login("alice", "alicepass")

Execute SQL:
SELECT * FROM users WHERE username = 'alice' AND password = 'alicepass'

✅ Success! Login with User: (2, 'alice', 'alicepass')


In [10]:
# Login Test 2
vulnerable_login("admin", "admin2025")

Execute SQL:
SELECT * FROM users WHERE username = 'admin' AND password = 'admin2025'

❌ Failed!


In [11]:
# SQL Injection Test with ( ' -- )
injection_username = "admin' --"
injection_password = "abc"

vulnerable_login(injection_username, injection_password)

Execute SQL:
SELECT * FROM users WHERE username = 'admin' --' AND password = 'abc'

✅ Success! Login with User: (1, 'admin', 'admin123')


In [12]:
#Create a Function - Safe Login
def safe_login(username, password):
    query = "SELECT * FROM users WHERE username = ? AND password = ?"
    print(f"Execute Safe SQL Query with Parameters.")

    cursor.execute(query, (username, password))
    result = cursor.fetchone()

    if result:
        print("✅ Success! Login with User:", result)
    else:
        print("❌ Failed.")

In [13]:
# SQL Injection Test with ( ' -- ) - Safe Function
safe_login("admin' --", "abc")

Execute Safe SQL Query with Parameters.
❌ Failed.
